# Red Team Evaluation - Generic Template

Starting point for integrating any target. The only requirement is an async `handler(prompt, history, session_id, target_system_prompt)` that calls your target and returns its reply as a string.

Common targets you can drop in:
- an OpenAI / LLM SDK call (see `../red_team_eval.ipynb`)
- an HTTP request to your own API (see `http_api.ipynb`)
- browser automation via Playwright (see `playwright.ipynb`)
- a CLI subprocess, gRPC call, etc.

**Prerequisites:**
- `pip install -r ../../requirements.txt`
- `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` in `../../.env`

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
import os

from dotenv import load_dotenv
from hiddenlayer import AsyncHiddenLayer

load_dotenv("../../.env")

hl_client = AsyncHiddenLayer(
    client_id=os.getenv("HIDDENLAYER_CLIENT_ID"),
    client_secret=os.getenv("HIDDENLAYER_CLIENT_SECRET"),
)

## Configurables

1. Configure target
2. Configure evaluation

In [ ]:
# Target configuration
TARGET_MODEL_NAME = "my-target-model"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."

# Evaluation configuration
EVAL_NAME = "Red Team Eval"  # Name of the evaluation
EXECUTION_STRATEGY = "single"  # Options: "single", "random", "static_prompt_set"
MAX_TURNS = 5  # Options: 1-5
SESSIONS_PER_TECHNIQUE = 1  # Options: 1-5
PARALLEL_TECHNIQUES = 5  # Options: 1-10
HIDDENLAYER_PROJECT_ID = None  # Options: None, <project_id>

## Create Handler

The handler acts as a proxy between the attacker and the target. Replace the body with any logic that calls your target and returns a string.

- `prompt`: the attack prompt generated by HiddenLayer for this turn
- `history`: prior turns as `[{"role": ..., "content": ...}, ...]`
- `session_id`: stable id for the current attack session
- `target_system_prompt`: system prompt the target should run with

In [ ]:
async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and target."""
    # TODO: replace this with a real call to your target.
    raise NotImplementedError(
        "Implement handler to call your target and return its reply."
    )

## Run the Evaluation

Open a red team session and run attack techniques in parallel against the target.

In [ ]:
session = await hl_client.evaluation_sessions.red_team.start_session(
    name=EVAL_NAME,
    target_model=TARGET_MODEL_NAME,
    target_system_prompt=TARGET_SYSTEM_PROMPT,
    execution_strategy_type=EXECUTION_STRATEGY,
    max_turns=MAX_TURNS,
    sessions_per_technique=SESSIONS_PER_TECHNIQUE,
    max_parallel_techniques=PARALLEL_TECHNIQUES,
    hiddenlayer_project_id=HIDDENLAYER_PROJECT_ID,
)

print(f"Session started: {session.workflow_id}")

await session.run_with_callback_parallel(handler=handler)

print("Evaluation complete. View transcripts at https://console.hiddenlayer.ai")

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await hl_client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")